In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from vllm import LLM, SamplingParams

from peft import LoraConfig, get_peft_model



INFO 03-17 10:50:12 __init__.py:183] Automatically detected platform cuda.


In [2]:

model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover-critic",
    device_map="cuda:1",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding(92544, 2048, padding_idx=2)

In [5]:

#
config = LoraConfig(
    target_modules=[
        "wqkv",
        "wo",
        "gate_up_proj",
        "w2", ],
    task_type='CAUSAL_LM',
    r=16,
    lora_alpha=1,
    lora_dropout=0.1,
)
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()


trainable params: 7,864,320 || all params: 1,707,446,272 || trainable%: 0.46058960266950055


In [3]:

config = model.config

In [4]:

config.architectures = ['InternLM2ForCausalLM']

In [5]:

config.auto_map = {"AutoConfig": "configuration_internlm2.InternLM2Config",
                   "AutoModel": "modeling_internlm2.InternLM2ForCausalLM"}

In [6]:

lm_model = AutoModel.from_pretrained("internlm/internlm2_5-step-prover-critic", trust_remote_code=True,
                                  torch_dtype=torch.float16, device_map="cuda:1",
                                  config=config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at internlm/internlm2_5-step-prover-critic were not used when initializing InternLM2ForCausalLM: {'v_head.weight'}
- This IS expected if you are initializing InternLM2ForCausalLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing InternLM2ForCausalLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of InternLM2ForCausalLM were not initialized from the model checkpoint at internlm/internlm2_5-step-prover-critic and are newly initialized: ['output.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
config = LoraConfig(
    target_modules=[
        "wqkv",
        "wo",
        "gate_up_proj",
        "w2",
    ],
    task_type='CAUSAL_LM',
    r=16,
    lora_alpha=1,
    lora_dropout=0.1,
)
lora_lm_model = get_peft_model(lm_model, config)

lora_lm_model.output.weight.requires_grad = True

lora_lm_model.print_trainable_parameters()

trainable params: 197,394,432 || all params: 1,896,974,336 || trainable%: 10.40575131955818


In [32]:

for n, a in lora_lm_model.named_parameters():
    print (n, a.requires_grad)

base_model.model.model.tok_embeddings.weight False
base_model.model.model.layers.0.attention.wqkv.weight False
base_model.model.model.layers.0.attention.wqkv.lora_A.default.weight True
base_model.model.model.layers.0.attention.wqkv.lora_B.default.weight True
base_model.model.model.layers.0.attention.wo.weight False
base_model.model.model.layers.0.attention.wo.lora_A.default.weight True
base_model.model.model.layers.0.attention.wo.lora_B.default.weight True
base_model.model.model.layers.0.feed_forward.w1.weight False
base_model.model.model.layers.0.feed_forward.w3.weight False
base_model.model.model.layers.0.feed_forward.w2.weight False
base_model.model.model.layers.0.feed_forward.w2.lora_A.default.weight True
base_model.model.model.layers.0.feed_forward.w2.lora_B.default.weight True
base_model.model.model.layers.0.attention_norm.weight False
base_model.model.model.layers.0.ffn_norm.weight False
base_model.model.model.layers.1.attention.wqkv.weight False
base_model.model.model.layers.1.

In [36]:

lora_lm_model.output.weight

torch.Size([92544, 2048])

In [205]:
model

InternLM2ForCausalLM(
  (model): InternLM2Model(
    (tok_embeddings): Embedding(92544, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x InternLM2DecoderLayer(
        (attention): InternLM2Attention(
          (wqkv): Linear(in_features=2048, out_features=4096, bias=False)
          (wo): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): InternLM2DynamicNTKScalingRotaryEmbedding()
        )
        (feed_forward): InternLM2MLP(
          (w1): Linear(in_features=2048, out_features=8192, bias=False)
          (w3): Linear(in_features=2048, out_features=8192, bias=False)
          (w2): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (attention_norm): InternLM2RMSNorm()
        (ffn_norm): InternLM2RMSNorm()
      )
    )
    (norm): InternLM2RMSNorm()
  )
  (output): Linear(in_features=2048, out_features=92544, bias=False)
)

ValueError: You have to specify either input_ids or inputs_embeds

In [7]:

tokenizer = AutoTokenizer.from_pretrained("internlm/internlm2_5-step-prover-critic", trust_remote_code=True)

chat_1 = [
    {"role": "user", "content": "Which state is closer to 'no goals'?"},
    {"role": "assistant", "content": "no goals"}
]
chat_2 = [
    {"role": "user", "content": "Which state is closer to 'no goals'?"},
    {"role": "assistant", "content": "x : ℕ\nh₀ : ↑x + 4 / 100 * ↑x = 598\n⊢ 100 * x = 100 * 575"}
]

score1 = model.get_score(tokenizer, chat_1)
score2 = model.get_score(tokenizer, chat_2)
print("score1: ", score1)
print("score2: ", score2)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

score1:  16.5
score2:  1.595703125


In [ ]:

model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover",
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

In [19]:

tokenizer = AutoTokenizer.from_pretrained("internlm/internlm2_5-step-prover", trust_remote_code=True)

In [2]:
llm = LLM(model='internlm/internlm2_5-step-prover', trust_remote_code=True, tensor_parallel_size=2)


INFO 02-14 15:30:04 config.py:134] Replacing legacy 'type' key with 'rope_type'
INFO 02-14 15:30:14 config.py:520] This model supports multiple tasks: {'classify', 'generate', 'reward', 'embed', 'score'}. Defaulting to 'generate'.
INFO 02-14 15:30:14 config.py:1328] Defaulting to use mp for distributed inference
INFO 02-14 15:30:14 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='internlm/internlm2_5-step-prover', speculative_config=None, tokenizer='internlm/internlm2_5-step-prover', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityC

Loading pt checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]


INFO 02-14 15:30:42 model_runner.py:1115] Loading model weights took 7.2095 GB
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:42 model_runner.py:1115] Loading model weights took 7.2095 GB
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:47 worker.py:266] Memory profiling takes 4.68 seconds
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:47 worker.py:266] the current vLLM instance can use total_gpu_memory (47.29GiB) x gpu_memory_utilization (0.90) = 42.56GiB
(VllmWorkerProcess pid=73005) INFO 02-14 15:30:47 worker.py:266] model weights take 7.21GiB; non_torch_memory takes 0.38GiB; PyTorch activation peak memory takes 0.65GiB; the rest of the memory reserved for KV Cache is 34.32GiB.
INFO 02-14 15:30:47 worker.py:266] Memory profiling takes 4.43 seconds
INFO 02-14 15:30:47 worker.py:266] the current vLLM instance can use total_gpu_memory (47.32GiB) x gpu_memory_utilization (0.90) = 42.59GiB
INFO 02-14 15:30:47 worker.py:266] model weights take 7.21GiB; non_torch_memory takes 0.41GiB; PyTorc

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:24<00:00,  1.41it/s]

INFO 02-14 15:31:14 custom_all_reduce.py:224] Registering 2275 cuda graph addresses
(VllmWorkerProcess pid=73005) INFO 02-14 15:31:14 custom_all_reduce.py:224] Registering 2275 cuda graph addresses
(VllmWorkerProcess pid=73005) INFO 02-14 15:31:15 model_runner.py:1558] Graph capturing finished in 25 secs, took 0.43 GiB
INFO 02-14 15:31:15 model_runner.py:1558] Graph capturing finished in 25 secs, took 0.43 GiB
INFO 02-14 15:31:15 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 32.08 seconds


In [3]:
prompt = [
    f"---\nNAME: {'square_sub_one_divisible_eight'}\n\n---\nPROOF_BEFORE: {'rw [h, pow_two]'}\n\n---\nSTATE_BEFORE: 'm n : N\nh : n = 2 * m + 1\n⊢ 8 | n * n - 1'\n\n---\nTACTIC: "]


In [49]:
sampling_params = SamplingParams(n=32, temperature=0.7, stop_token_ids=[92542], best_of=32, logprobs=0)  #, top_p=0.95)
out = llm.generate(prompt, sampling_params)



Processed prompts:   3%|▎         | 1/32 [00:00<00:20,  1.54it/s, est. speed input: 103.16 toks/s, output: 538.87 toks/s]


In [51]:

[(i.text.strip(), i.cumulative_logprob) for i in out[0].outputs]

[('rw [h, mul_add, mul_one]', -1.960043552557181),
 ('rw [h, mul_add, mul_one]', -1.960043552557181),
 ('rw [h, mul_add, mul_one]', -1.960043552557181),
 ('rw [h]', -2.105288189071871),
 ('rw [h]', -2.105288189071871),
 ('rw [h]', -2.105288189071871),
 ('rw [h, mul_add, mul_one, add_comm]', -2.1188550177248544),
 ('rw [h, mul_add, mul_one, add_comm]', -2.1188550177248544),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, Nat.mul_add, Nat.mul_one]', -2.391726188005123),
 ('rw [h, pow_two]', -2.799261191441474),
 ('rw [h, pow_two]', -2.799261191441474),
 ('rw [h, pow_two]', -2.799261191441474),
 ('rw [h, pow_two]', -2.799261191441474),
 ('simp [h, Nat.mul_mod, Nat.add_mod, Nat.mod_mod]', -2.840807825133652),
 ('ring_nf', -3.2021668000525096),
 ('ring_nf', -3.2021668000525096),
 ('ring_nf', -3.2021668000525096),
 ('rw [h, add_mul, one_mul]', -3.72579727

In [45]:

out[0].outputs[0]


CompletionOutput(index=0, text='rw [h, mul_add, mul_one]\n\n', token_ids=(31948, 640, 280, 328, 15734, 3054, 328, 15734, 11817, 2690, 92542), cumulative_logprob=-1.955109274473216, logprobs=[{31948: Logprob(logprob=-0.31204405426979065, rank=1, decoded_token='rw')}, {640: Logprob(logprob=-8.010543388081715e-05, rank=1, decoded_token=' [')}, {280: Logprob(logprob=-0.0015279296785593033, rank=1, decoded_token='h')}, {328: Logprob(logprob=-0.184807687997818, rank=1, decoded_token=',')}, {15734: Logprob(logprob=-0.6534253358840942, rank=1, decoded_token=' mul')}, {3054: Logprob(logprob=-0.019588593393564224, rank=1, decoded_token='_add')}, {328: Logprob(logprob=-0.02790931798517704, rank=1, decoded_token=',')}, {15734: Logprob(logprob=-0.05755041539669037, rank=1, decoded_token=' mul')}, {11817: Logprob(logprob=-0.004202107898890972, rank=1, decoded_token='_one')}, {2690: Logprob(logprob=-0.6936477422714233, rank=1, decoded_token=']\n\n'), 328: Logprob(logprob=-0.6936477422714233, rank=1, 

In [ ]:
print(prompt)

In [ ]:

tokenized_state = tokenizer(
    prompt,
    padding="longest",
    max_length=2000,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()

In [ ]:
out = model.generate(
    input_ids=state_ids,
    # max_new_tokens=4,
    num_return_sequences=64,
    do_sample=True,
    output_scores=True,
    return_dict_in_generate=True,
    temperature=0.7
)

# Return the output.
raw_output_text = tokenizer.batch_decode(
    out.sequences, skip_special_tokens=False
)


In [ ]:

raw_output_text

In [15]:

data = [{
    'goal': 'What is the result of applying TACTICS to STATE? ---\n' +
            'NAME: amc12a_2009_p6\n' +
            '\n' +
            '---\n' +
            'PROOF_BEFORE: \n' +
            '\n' +
            '---\n' +
            'STATE: m n p q : ℝ\n' +
            'h₀ : p = 2 ^ m\n' +
            'h₁ : q = 3 ^ n\n' +
            '⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n' +
            '\n' +
            '---\n' +
            'TACTICS: \n' +
            '\n',
    'theorem': 'amc12a_2009_p6',
    'split': 'train',
    'tactic': 'subst h₀ h₁',
    'result': 'm n : ℝ\n⊢ (2 ^ m) ^ (2 * n) * (3 ^ n) ^ m = 12 ^ (m * n)',
    'status': 'success\n',
    'goal_score': -1.7509765625,
    'tac_index': 9,
    'all_tacs': [
        'field_simp [h₀, h₁]',
        'simp [h₀, h₁, pow_mul, pow',
        'simp only [h₀, h₁, mul_rpow',
        'simp [h₀, h₁, mul_comm]',
        'rw [h₀, h₁, ← Real.r',
        'rw [h₀, h₁]',
        'rw [h₀, h₁, ← pow_mul',
        'rw [h₀, h₁] <;> ring',
        'simp only [h₀, h₁]',
        'subst h₀ h₁',
        'simp [h₀, h₁, mul_comm, mul'
    ],
    'rand_idx': 0.8926771765450751
},
    {
        'goal': 'What is the result of applying TACTICS to STATE? ---\n' +
                'NAME: amc12a_2009_p6\n' +
                '\n' +
                '---\n' +
                'PROOF_BEFORE: \n' +
                '\n' +
                '---\n' +
                'STATE: m n p q : ℝ\n' +
                'h₀ : p = 2 ^ m\n' +
                'h₁ : q = 3 ^ n\n' +
                '⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n' +
                '\n' +
                '---\n' +
                'TACTICS: \n' +
                '\n',
        'theorem': 'amc12a_2009_p6',
        'split': 'train',
        'tactic': 'rw [h₀, h₁] <;> ring',
        'result': 'm n p q : ℝ\n' +
                  'h₀ : p = 2 ^ m\n' +
                  'h₁ : q = 3 ^ n\n' +
                  '⊢ (2 ^ m) ^ (n * 2) * (3 ^ n) ^ m = 12 ^ (n * m)',
        'status': 'success\n',
        'goal_score': -1.3935546875,
        'tac_index': 7,
        'all_tacs': [
            'field_simp [h₀, h₁]',
            'simp [h₀, h₁, pow_mul, pow',
            'simp only [h₀, h₁, mul_rpow',
            'simp [h₀, h₁, mul_comm]',
            'rw [h₀, h₁, ← Real.r',
            'rw [h₀, h₁]',
            'rw [h₀, h₁, ← pow_mul',
            'rw [h₀, h₁] <;> ring',
            'simp only [h₀, h₁]',
            'subst h₀ h₁',
            'simp [h₀, h₁, mul_comm, mul'
        ],
        'rand_idx': 0.6408123687564706
    }, ]


In [16]:

tokenised_up_to_target = [
    'THEOREM:\n' + ex['theorem'] + '\n' + ex['goal'] + '\n'.join([t for t in ex['all_tacs'][:ex['tac_index']]]) + '\n'
    for ex in data]

In [17]:

tokenised_up_to_target

['THEOREM:\namc12a_2009_p6\nWhat is the result of applying TACTICS to STATE? ---\nNAME: amc12a_2009_p6\n\n---\nPROOF_BEFORE: \n\n---\nSTATE: m n p q : ℝ\nh₀ : p = 2 ^ m\nh₁ : q = 3 ^ n\n⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n\n---\nTACTICS: \n\nfield_simp [h₀, h₁]\nsimp [h₀, h₁, pow_mul, pow\nsimp only [h₀, h₁, mul_rpow\nsimp [h₀, h₁, mul_comm]\nrw [h₀, h₁, ← Real.r\nrw [h₀, h₁]\nrw [h₀, h₁, ← pow_mul\nrw [h₀, h₁] <;> ring\nsimp only [h₀, h₁]\n',
 'THEOREM:\namc12a_2009_p6\nWhat is the result of applying TACTICS to STATE? ---\nNAME: amc12a_2009_p6\n\n---\nPROOF_BEFORE: \n\n---\nSTATE: m n p q : ℝ\nh₀ : p = 2 ^ m\nh₁ : q = 3 ^ n\n⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n\n---\nTACTICS: \n\nfield_simp [h₀, h₁]\nsimp [h₀, h₁, pow_mul, pow\nsimp only [h₀, h₁, mul_rpow\nsimp [h₀, h₁, mul_comm]\nrw [h₀, h₁, ← Real.r\nrw [h₀, h₁]\nrw [h₀, h₁, ← pow_mul\n']

In [20]:

# todo get location of tactic tokens in tokenised goal

tokenised_up_to_target = tokenizer(tokenised_up_to_target,
                                   padding='longest',
                                   max_length=2000,
                                   truncation=True, return_tensors='pt')


In [21]:

tokenised_up_to_target

{'input_ids': tensor([[    1, 17368,  6035,   307,   642,   440,   271,   845,   264,   284,
          1174,   305,   752,   321,   364,  3993,   505,   410,  1245,   446,
         19105,   481,  6974, 19258,   442, 22710,   345, 52959,  7677,   334,
          1221,   271,   845,   264,   284,  1174,   305,   752,   321,   402,
         11102,  9246, 12634, 27566, 33128,   334,  4872, 11102, 25046,   334,
           427,   439,   412,  2968,   680,   262,   229,   135,   160,   364,
           280,   229,   133,   131,   680,   412,   415,   262,   314,  6463,
           427,   364,   280,   229,   133,   132,   680,  2968,   415,   262,
           308,  6463,   439,   364,   229,   141,   165,   412,  6463,   451,
           314,   484,   439,   313,   484,  2968,  6463,   427,   415,   262,
           845,  6463,   451,   277,   484,   439,   824, 11102,   291,  6974,
         19258,   334,  4872,  2725,   774,  6815,   640,   280,   229,   133,
           131,   328,   436,   229,  

In [121]:

lens_before = tokenised_up_to_target.attention_mask.sum(dim=1)

In [122]:

# lens_before = torch.tensor([143, 252])
lens_before

tensor([252, 222])

In [125]:


target_tactics = [ex['tactic'] for ex in data]

In [126]:

target_tactics

['subst h₀ h₁', 'rw [h₀, h₁] <;> ring']

In [127]:

tokenized_tactics = tokenizer(
    target_tactics,
    padding="longest",
    max_length=2000,
    truncation=True,
    return_tensors="pt",
)

In [128]:

tac_lens = tokenized_tactics.attention_mask.sum(dim=1)

In [129]:

target_inds = (lens_before, tac_lens)

In [130]:

target_inds

(tensor([252, 222]), tensor([10, 17]))

In [131]:


goal = ['THEOREM:\n' + ex['theorem'] + '\n' + ex['goal'] + '\n'.join(ex['all_tacs']) for ex in data]

In [132]:

goal

['THEOREM:\namc12a_2009_p6\nWhat is the result of applying TACTICS to STATE? ---\nNAME: amc12a_2009_p6\n\n---\nPROOF_BEFORE: \n\n---\nSTATE: m n p q : ℝ\nh₀ : p = 2 ^ m\nh₁ : q = 3 ^ n\n⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n\n---\nTACTICS: \n\nfield_simp [h₀, h₁]\nsimp [h₀, h₁, pow_mul, pow\nsimp only [h₀, h₁, mul_rpow\nsimp [h₀, h₁, mul_comm]\nrw [h₀, h₁, ← Real.r\nrw [h₀, h₁]\nrw [h₀, h₁, ← pow_mul\nrw [h₀, h₁] <;> ring\nsimp only [h₀, h₁]\nsubst h₀ h₁\nsimp [h₀, h₁, mul_comm, mul',
 'THEOREM:\namc12a_2009_p6\nWhat is the result of applying TACTICS to STATE? ---\nNAME: amc12a_2009_p6\n\n---\nPROOF_BEFORE: \n\n---\nSTATE: m n p q : ℝ\nh₀ : p = 2 ^ m\nh₁ : q = 3 ^ n\n⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n\n---\nTACTICS: \n\nfield_simp [h₀, h₁]\nsimp [h₀, h₁, pow_mul, pow\nsimp only [h₀, h₁, mul_rpow\nsimp [h₀, h₁, mul_comm]\nrw [h₀, h₁, ← Real.r\nrw [h₀, h₁]\nrw [h₀, h₁, ← pow_mul\nrw [h₀, h₁] <;> ring\nsimp only [h₀, h₁]\nsubst h₀ h₁\nsimp [h₀, h₁, mul_comm, mul']

In [133]:


tokenized_goal = tokenizer(
    goal,
    padding="longest",
    max_length=2000,
    truncation=True,
    return_tensors="pt",
)

In [134]:

tokenized_goal.input_ids.shape


torch.Size([2, 278])

In [135]:
tac_tokens = [tokenized_goal.input_ids[i][target_inds[0][i]:target_inds[0][i] + target_inds[1][i] - 1] for i in
              range(len(data))]

In [136]:
tac_tokens

[tensor([51887,   436,   229,   133,   131,   436,   229,   133,   132]),
 tensor([31948,   640,   280,   229,   133,   131,   328,   436,   229,   133,
           132,   332,   497,   329,   330, 10195])]

In [137]:
tokenizer.batch_decode(tac_tokens)

['subst h₀ h₁', 'rw [h₀, h₁] <;> ring']

In [144]:

output = model.model.forward(tokenized_goal.input_ids.cuda(), attention_mask=tokenized_goal.attention_mask.cuda())

In [166]:

test = model.generate(tokenized_goal.input_ids.cuda(), attention_mask=tokenized_goal.attention_mask.cuda())

TypeError: The current model class (InternLM2ForRewardModel) is not compatible with `.generate()`, as it doesn't have a language model head. Classes that support generation often end in one of these names: ['ForCausalLM', 'ForConditionalGeneration', 'ForSpeechSeq2Seq', 'ForVision2Seq'].

In [222]:
output_ = model.forward(tokenized_goal.input_ids.cuda(), attention_mask=tokenized_goal.attention_mask.cuda())


In [218]:

output[0].shape

torch.Size([2, 278, 2048])

In [228]:

output_[1]

((tensor([[[[-1.8311e-01, -1.3220e-01, -2.7783e-01,  ..., -8.5107e-01,
             -6.2354e-01, -6.3904e-02],
            [ 5.5176e-01, -1.9912e+00, -2.0938e+00,  ..., -1.2646e+00,
             -2.3496e+00,  3.7969e+00],
            [ 2.9844e+00, -2.3438e+00,  7.0703e-01,  ..., -1.3018e+00,
             -1.9453e+00,  2.2676e+00],
            ...,
            [-3.3613e+00, -2.7246e+00,  1.7559e+00,  ..., -1.4534e-03,
             -8.5156e-01,  2.3809e+00],
            [-9.8145e-02, -3.2422e-01,  4.1504e-01,  ..., -1.9111e+00,
             -2.3203e+00,  1.2285e+00],
            [ 6.4062e-01,  2.7656e+00,  4.1846e-01,  ...,  1.1963e+00,
             -5.7861e-01,  8.8043e-03]],
  
           [[-1.6760e-01, -3.2910e-01,  3.9764e-02,  ...,  1.7773e+00,
              1.9512e+00,  2.9150e-01],
            [ 2.3193e-01, -1.0420e+00, -8.6719e-01,  ...,  3.2007e-01,
              8.4619e-01, -2.2383e+00],
            [ 4.2578e+00, -5.5420e-01, -2.7363e+00,  ...,  5.6348e-01,
              2.1172

In [35]:

result = [ex['status'] + ex["result"] for ex in examples]

tokenized_result = self.tokenizer(
    result,
    padding="longest",
    max_length=self.max_seq_len - tokenized_goal.input_ids.shape[1],
    truncation=True,
    return_tensors="pt",
)

# print (tokenized_goal.input_ids.shape, tokenized_result.input_ids.shape)


# result_ids = tokenized_result.input_ids

# result_ids[result_ids == self.tokenizer.pad_token_id] = -100  # todo equivalent token id for internlm?

batch = {}
batch["goal"] = goal
batch["goal_ids"] = tokenized_goal.input_ids
batch["goal_mask"] = tokenized_goal.attention_mask
batch["result"] = result
batch["result_ids"] = tokenized_result.input_ids
batch["result_mask"] = tokenized_goal.attention_mask
batch["tactic"] = target_tactics
batch["target_inds"] = target_inds
batch["status"] = [ex['status'] for ex in examples]

# # Copy other fields.


NameError: name 'examples' is not defined

In [206]:
enc_model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover-critic",
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

config = model.config

config.architectures = ['InternLM2ForCausalLM']

config.auto_map = {"AutoConfig": "configuration_internlm2.InternLM2Config",
                   "AutoModel": "modeling_internlm2.InternLM2ForCausalLM"}

lm_model = AutoModel.from_pretrained("internlm/internlm2_5-step-prover-critic", trust_remote_code=True,
                                     torch_dtype=torch.float16, device_map="cuda",
                                     config=config)




Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at internlm/internlm2_5-step-prover-critic were not used when initializing InternLM2ForCausalLM: {'v_head.weight'}
- This IS expected if you are initializing InternLM2ForCausalLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing InternLM2ForCausalLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of InternLM2ForCausalLM were not initialized from the model checkpoint at internlm/internlm2_5-step-prover-critic and are newly initialized: ['output.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [210]:

enc_model.model

# run through enc_model, get indices from last_hidden_state (batch_size x seq_len x emb_dim),
# get indices based on target_inds, mean pool for tactic embedding,


# using decoder model, give tactic embedding + original goal with inputs_embeds
# todo how to deal with labels... manually have to get logits for each token in output and calculate CE loss?

InternLM2Model(
  (tok_embeddings): Embedding(92544, 2048, padding_idx=2)
  (layers): ModuleList(
    (0-23): 24 x InternLM2DecoderLayer(
      (attention): InternLM2Attention(
        (wqkv): Linear(in_features=2048, out_features=4096, bias=False)
        (wo): Linear(in_features=2048, out_features=2048, bias=False)
        (rotary_emb): InternLM2DynamicNTKScalingRotaryEmbedding()
      )
      (feed_forward): InternLM2MLP(
        (w1): Linear(in_features=2048, out_features=8192, bias=False)
        (w3): Linear(in_features=2048, out_features=8192, bias=False)
        (w2): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (attention_norm): InternLM2RMSNorm()
      (ffn_norm): InternLM2RMSNorm()
    )
  )
  (norm): InternLM2RMSNorm()
)

In [231]:

lm_out = lm_model.forward(input_ids=tokenized_goal.input_ids.cuda(),
                          attention_mask=tokenized_goal.attention_mask.cuda())

In [282]:

lm_model.model.tok_embeddings.

Embedding(92544, 2048, padding_idx=2)

In [273]:
lm_out.logits.shape


torch.Size([2, 278, 92544])

In [247]:
lm_out.logits[..., :-1, :].contiguous()[0][100]

# Shift so that tokens < n predict n
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()
# Flatten the tokens
loss_fct = CrossEntropyLoss()
shift_logits = shift_logits.view(-1, self.config.vocab_size)
shift_labels = shift_labels.view(-1)
# Enable model parallelism
shift_labels = shift_labels.to(shift_logits.device)
loss = loss_fct(shift_logits, shift_labels)

tensor([ 7.8242, -4.1836, -0.7646,  ..., -1.8115,  2.1523, -4.6094],
       device='cuda:0', grad_fn=<SelectBackward0>)

In [264]:

torch.index_select(lm_model.model.tok_embeddings, 0, torch.tensor([1, 2, 3]), dim=0)

TypeError: index_select() received an invalid combination of arguments - got (Embedding, int, Tensor, dim=int), but expected one of:
 * (Tensor input, int dim, Tensor index, *, Tensor out = None)
 * (Tensor input, name dim, Tensor index, *, Tensor out = None)


In [272]:

print(lm_model.model.tok_embeddings(torch.tensor([1, 2, 3]).cuda()).shape)

torch.Size([3, 2048])


In [14]:


lora_lm_model.model

InternLM2ForCausalLM(
  (model): InternLM2Model(
    (tok_embeddings): Embedding(92544, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x InternLM2DecoderLayer(
        (attention): InternLM2Attention(
          (wqkv): Linear(
            in_features=2048, out_features=4096, bias=False
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
          )
          (wo): Linear(
            in_features=2048, out_features=2048, bias=False
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict

In [23]:
    lora_lm_model.generate(input_ids=tokenised_up_to_target.input_ids.to('cuda:1'),


                                       max_length=3000,
                                       num_beams=4,
                                       do_sample=False,
                                       num_return_sequences=4,
                                       early_stopping=True,
                                       output_scores=True,
                                       return_dict_in_generate=True,
                                       )

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


KeyboardInterrupt: 